In [3]:
import pandas as pd

#df = pd.read_csv("data/teams.csv")
df = pd.read_csv("teams_2026.csv")

# Clean team names
df["team"] = (
    df["team"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.replace("*", "", regex=False)
    .str.strip()
)

df["team"] = df["team"].replace({"Albany (NY)": "Albany"})

# Remove accidental repeated headers if they exist
df = df[df["team"] != "School"].copy()

# Coerce numeric columns
num_cols = df.columns.drop("team")
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# Drop separator columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Rename repeated columns to match Sports-Reference structure
df = df.rename(columns={
    "wins": "W",
    "losses": "L",
    "wins.1": "Conf_W",
    "losses.1": "Conf_L",
    "wins.2": "Home_W",
    "losses.2": "Home_L",
    "wins.3": "Away_W",
    "losses.3": "Away_L",
    "Tm.": "Points_For",
    "Opp.": "Points_Against",
})
print(df.columns)
print(df.head())

Index(['Rk', 'team', 'G', 'W', 'L', 'win_pct', 'SRS', 'SOS', 'Conf_W',
       'Conf_L', 'Home_W', 'Home_L', 'Away_W', 'Away_L', 'Points_For',
       'Points_Against', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', 'FT',
       'FTA', 'FT%', 'ORB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'year'],
      dtype='object')
    Rk               team     G     W     L  win_pct    SRS    SOS  Conf_W  \
0  1.0  Abilene Christian  33.0  14.0  19.0    0.424  -7.38  -1.45     5.0   
1  2.0          Air Force  32.0   3.0  29.0    0.094 -13.75   4.60     0.0   
2  3.0              Akron  35.0  29.0   6.0    0.829   8.48  -2.91    17.0   
3  4.0            Alabama  35.0  25.0  10.0    0.714  23.03  14.57    13.0   
4  5.0        Alabama A&M  33.0  18.0  15.0    0.545 -13.31 -10.81    10.0   

   Conf_L  ...    FTA    FT%    ORB     TRB    AST    STL    BLK    TOV  \
0    13.0  ...  741.0  0.704  369.0  1047.0  440.0  321.0   91.0  472.0   
1    20.0  ...  574.0  0.641  225.0   931.0  389.0  180.0   76

In [4]:
########################################
# 1. LOAD AND CLEAN TEAM SEASON STATS
########################################

#df = pd.read_csv("data/teams.csv")
df = pd.read_csv("teams_2026.csv")

# Clean team names
df["team"] = (
    df["team"]
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.replace("*", "", regex=False)
    .str.strip()
)

# Normalize known name variants
df["team"] = df["team"].replace({
    "Albany (NY)": "Albany"
})

# Remove accidental header rows
df = df[df["team"] != "School"].copy()

# Coerce numeric columns
num_cols = df.columns.drop("team")
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# Drop separator columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Rename repeated columns
df = df.rename(columns={
    "wins": "W",
    "losses": "L",
    "wins.1": "Conf_W",
    "losses.1": "Conf_L",
    "wins.2": "Home_W",
    "losses.2": "Home_L",
    "wins.3": "Away_W",
    "losses.3": "Away_L",
    "Tm.": "Points_For",
    "Opp.": "Points_Against",
})

########################################
# 2. LOAD TOURNAMENT GAMES
########################################

games = pd.read_csv("data/tournament_games.csv")
games = pd.read_csv("tournament_games_2026.csv")

# Normalize team names in tournament data
games["winner"] = games["winner"].replace({
    "Albany (NY)": "Albany"
})
games["loser"] = games["loser"].replace({
    "Albany (NY)": "Albany"
})

########################################
# 3. MERGE TEAM STATS (WINNER + LOSER)
########################################

# Merge winner stats
games_w = games.merge(
    df,
    left_on=["winner", "season"],
    right_on=["team", "year"],
    how="left"
)

# Merge loser stats
games_wl = games_w.merge(
    df,
    left_on=["loser", "season"],
    right_on=["team", "year"],
    how="left",
    suffixes=("_winner", "_loser")
)

########################################
# 4. CREATE DIFFERENCE FEATURES
########################################

features = [
    "SRS",
    "win_pct",
    "FG%",
    "3P%",
    "FT%",
    "TRB",
    "AST",
    "TOV"
]

for col in features:
    games_wl[f"{col}_diff"] = (
        games_wl[f"{col}_winner"] -
        games_wl[f"{col}_loser"]
    )

# Seed difference (lower seed = stronger team)
games_wl["seed_diff"] = (
    games_wl["winner_seed"] -
    games_wl["loser_seed"]
)

########################################
# 5. CREATE SYMMETRIC DATASET
########################################

# Winner perspective
games_wl["win_label"] = 1

# Loser perspective
loser_view = games_wl.copy()

for col in features:
    loser_view[f"{col}_diff"] *= -1

loser_view["seed_diff"] *= -1
loser_view["win_label"] = 0

# Combine both perspectives
model_df = pd.concat([games_wl, loser_view], ignore_index=True)

########################################
# 6. BUILD FINAL ML DATASET
########################################

# Select difference features (remove duplicates safely)
feature_cols = sorted(set(
    col for col in model_df.columns
    if col.endswith("_diff")
))

final_df = model_df[feature_cols + ["win_label", "season"]]

# Drop rows with failed merges
final_df = final_df.dropna()

########################################
# 7. FINAL SANITY CHECKS
########################################

print("Final dataset shape:", final_df.shape)
print("Total NaNs:", final_df.isna().sum().sum())
print(final_df.head())

########################################
# 8. SAVE TO CSV
########################################

final_df.to_csv("data/tournament_model_ml_2026.csv", index=False)

Final dataset shape: (100, 11)
Total NaNs: 0
   3P%_diff  AST_diff  FG%_diff  FT%_diff  SRS_diff  TOV_diff  TRB_diff  \
0     0.041     163.0     0.036    -0.042     33.79      48.0     328.0   
2    -0.017      63.0    -0.022     0.028     14.81      45.0     275.0   
3     0.012     142.0     0.022     0.054     17.86     -40.0       7.0   
4     0.034      16.0     0.033     0.015      9.02      40.0    -105.0   
5     0.001     111.0     0.005     0.041     21.33      27.0     126.0   

   seed_diff  win_pct_diff  win_label  season  
0        -15         0.264          1    2026  
2         -7         0.172          1    2026  
3         -9        -0.049          1    2026  
4         -5        -0.049          1    2026  
5        -11         0.000          1    2026  
